# LLM Instruction Fine-Tuning with LoRA

This notebook fine-tunes a small open-source instruction/chat model using **LoRA** (via `peft`) on a HuggingFace dataset, using `trl`'s `SFTTrainer`.

**Defaults used in this notebook** (feel free to change in the config cell):
- Base model: `Qwen/Qwen2.5-0.5B-Instruct` (small, fast, runs fine on a free Colab T4 GPU)
- Dataset: `mlabonne/guanaco-llama2-1k` (1k instruction/chat examples, already formatted as text — good for a quick demo)

**Before you start:** In Colab, go to `Runtime > Change runtime type` and select a **GPU** (T4 is fine).

Swap the model or dataset in the config cell to fine-tune on your own data — just point `DATASET_NAME` at any HuggingFace dataset repo id.

## 1. Install dependencies

In [1]:
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 99.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.9/842.9 kB 60.1 MB/s eta 0:00:00


## 2. (Optional) Log in to Hugging Face
Needed if your model/dataset is gated, or if you want to push the fine-tuned model to the Hub at the end.

In [2]:
from huggingface_hub import notebook_login

# Uncomment to log in:
# notebook_login()

## 3. Config — edit these to change model/dataset

In [3]:
# ==== Model & dataset ====
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"     # any causal LM on the Hub
DATASET_NAME = "mlabonne/guanaco-llama2-1k"    # any HF dataset repo id
DATASET_SPLIT = "train"
TEXT_FIELD = "text"                             # column containing the formatted text (set to None to auto-format below)

# ==== Output ====
OUTPUT_DIR = "./lora-finetuned-model"
PUSH_TO_HUB = False
HUB_MODEL_ID = "your-username/your-model-name"  # only used if PUSH_TO_HUB=True

# ==== LoRA config ====
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

# ==== Training config ====
NUM_EPOCHS = 1
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 2e-4
MAX_SEQ_LENGTH = 1024
USE_4BIT = True   # 4-bit quantized loading (QLoRA) — saves memory, good for free-tier GPUs

## 4. Load the dataset from Hugging Face

In [4]:
from datasets import load_dataset

dataset = load_dataset(DATASET_NAME, split=DATASET_SPLIT)
print(dataset)
print("\nExample record:")
print(dataset[0])

README.md:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

data/train-00000-of-00001-9ad84bb9cf65a4(…): reconstructing file:   0%|          |  0.00B /  967kB            

data/train-00000-of-00001-9ad84bb9cf65a4(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 1000
})

Example record:
{'text': '<s>[INST] Me gradué hace poco de la carrera de medicina ¿Me podrías aconsejar para conseguir rápidamente un puesto de trabajo? [/INST] Esto vale tanto para médicos como para cualquier otra profesión tras finalizar los estudios aniversarios y mi consejo sería preguntar a cuántas personas haya conocido mejor. En este caso, mi primera opción sería hablar con otros profesionales médicos, echar currículos en hospitales y cualquier centro de salud. En paralelo, trabajaría por mejorar mi marca personal como médico mediante un blog o formas digitales de comunicación como los vídeos. Y, para mejorar las posibilidades de encontrar trabajo, también participaría en congresos y encuentros para conseguir más contactos. Y, además de todo lo anterior, seguiría estudiando para presentarme a las oposiciones y ejercer la medicina en el sector público de mi país. </s>'}


### If your dataset isn't already a single text field

Many instruction datasets ship as separate `instruction` / `input` / `output` (or `prompt` / `response`) columns instead of one pre-formatted `text` column. If `TEXT_FIELD` above doesn't exist in your dataset, run the cell below to build one using the model's chat template.

In [5]:
from transformers import AutoTokenizer

_tok_for_formatting = AutoTokenizer.from_pretrained(MODEL_NAME)

def format_example(example):
    # Adjust these column names to match your dataset's actual columns
    instruction = example.get("instruction") or example.get("prompt") or ""
    response = example.get("output") or example.get("response") or ""
    messages = [
        {"role": "user", "content": instruction},
        {"role": "assistant", "content": response},
    ]
    example["text"] = _tok_for_formatting.apply_chat_template(messages, tokenize=False)
    return example

if TEXT_FIELD not in dataset.column_names:
    dataset = dataset.map(format_example)
    TEXT_FIELD = "text"
    print(dataset[0]["text"])
else:
    print(f"Dataset already has a '{TEXT_FIELD}' column — skipping reformatting.")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Dataset already has a 'text' column — skipping reformatting.


## 5. Load the base model (4-bit) and tokenizer

In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = None
if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16 if not USE_4BIT else None,
)
model.config.use_cache = False

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

## 6. Attach a LoRA adapter

In [7]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if USE_4BIT:
    model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359


## 7. Train with `SFTTrainer`

In [8]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,
    max_length=MAX_SEQ_LENGTH,
    dataset_text_field=TEXT_FIELD,
    report_to="none",
    push_to_hub=PUSH_TO_HUB,
    hub_model_id=HUB_MODEL_ID if PUSH_TO_HUB else None,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss
10,1.891426
20,1.881242
30,1.856773
40,1.860997
50,1.878518
60,1.777753
70,1.801818
80,1.710320


Step,Training Loss
10,1.891426
20,1.881242
30,1.856773
40,1.860997
50,1.878518
60,1.777753
70,1.801818
80,1.710320
90,1.813018
100,1.859353


TrainOutput(global_step=125, training_loss=1.8225221328735353, metrics={'train_runtime': 822.8737, 'train_samples_per_second': 1.215, 'train_steps_per_second': 0.152, 'total_flos': 1089070140787200.0, 'train_loss': 1.8225221328735353, 'entropy': 1.7222534835338592, 'num_tokens': 369206.0, 'mean_token_accuracy': 0.614980049431324, 'epoch': 1.0})

## 8. Save (and optionally push) the fine-tuned adapter

In [9]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved LoRA adapter + tokenizer to {OUTPUT_DIR}")

if PUSH_TO_HUB:
    trainer.push_to_hub()
    print(f"Pushed to https://huggingface.co/{HUB_MODEL_ID}")

Saved LoRA adapter + tokenizer to ./lora-finetuned-model


## 9. Quick test — run inference with the fine-tuned model

In [10]:
from transformers import pipeline

prompt_messages = [{"role": "user", "content": "Write a short poem about the ocean."}]
prompt = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tokens=200, do_sample=True, temperature=0.7)
print(tokenizer.decode(output[0], skip_special_tokens=True))

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in Qwen2DecoderLayer. Setting `past_key_values=None`.


system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Write a short poem about the ocean.
assistant
The>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[>[


## Notes / next steps
- **Change the model**: set `MODEL_NAME` to any causal LM on the Hub (e.g. `mistralai/Mistral-7B-Instruct-v0.3`, `meta-llama/Llama-3.2-1B-Instruct`). Larger models may need a bigger GPU (Colab Pro / A100) and/or a smaller batch size.
- **Change the dataset**: set `DATASET_NAME` to any dataset repo id on the Hub. If it doesn't have a pre-formatted text column, the notebook will build one from `instruction`/`output`-style columns in step 4 — adjust the column names there to match your dataset's schema.
- **Gated models/datasets** (like Llama) require you to accept the license on the Hub and log in via `notebook_login()` in step 2.
- **Merging the adapter**: to get a standalone model instead of a base model + adapter, use `model.merge_and_unload()` after training, then save the merged model.